# OptimusPrimus: training, inference and detection model

Run from the project root (the folder containing `Data/`). Every expensive cell is behind a flag set to `False`.

| step | what | section |
|---|---|---|
| acquire, annotate a subset | microscope TIFFs → tiles; CVAT annotations → masks | 0 |
| train | `train_all`: segmentation → automatic patches → classifier, each stored with its descriptor | 1 |
| *(alternative)* | segmentation → export patches → sort by hand → classifier | 1b |
| inference | `run_inference(images, seg_model, cls_model=None)`; `cls_model=None` is a segmentation-only run | 2 |
| descriptor and detection model | binned recall, precision, FP rate, FN, length bias; $\mu_j=\sum_i M_{ji}R_iN_i+\phi_jA$; comparison with the measurement | 3 |

Each z-stack is analysed on its **sharpest focal plane**, as the annotated tiles are.

**Model folders.** A segmentation model lives in `Data/models/<SEG_NAME>/`, where the name summarises how it was trained, e.g. `seg_ARCI-URAS26F2_n150-3fa2c1_MAnet-efficientnet-b7_ep200-bs2-lr3e-04-Dice-F1` (annotated set, number of tiles and hash of the file list, architecture, epochs, batch, learning rate, monitored metric). Next to `<SEG_NAME>.pth`:
- `<SEG_NAME>.json`: the full training record (files and train/val/test split, all hyperparameters, operating threshold, best epoch);
- `<SEG_NAME>_st<t>_descriptor.csv` and `_detection_logs.json`: the segmentation-only descriptor at threshold `t`;
- one `<SEG_NAME>__<CLS_TAG>.pth` (+ `.json`, descriptor) per classifier trained on it, e.g. `__cls-auto_pt0.1_efficientnet-b0_ep80-bs64-lr3e-05-pw0.7` (patch source and threshold, network, hyperparameters).

`OptimusPrimus.list_models()` lists them all.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import OptimusPrimus
from mineral_utils import slice_spectrum

## 0. Data preparation

Once per acquisition: TIFFs → 1024×1024 PNG tiles named `Frame<stack>_<acquisition>_<i>_<j>.png` (consecutive files form a z-stack); CVAT mask annotations (label `Track`) of the annotated subset → `<tile>_mask.png` in the `training_masks` subfolder of the annotated tiles.

In [ ]:
DATA_PREP = False

if DATA_PREP:
    OptimusPrimus.slice_tif_to_png_tiles(images_per_group=10, input_dir="Data/images/ARCI URAS26F2")   # planes per z-stack
    OptimusPrimus.convert_xml_to_masks("Data/annotations/annotations.xml", "Data/images/ARCI URAS26F2/training_masks")

## 1. Training

`train_all` trains the segmentation network, cuts the classifier patches automatically and trains the classifier on them:
every segmentation candidate of the train/val tiles is matched to the annotations (Hungarian IoU ≥ 0.5, the matching of the descriptor) and labelled track if matched, background otherwise. The patches are therefore exactly those the classifier sees at inference.

At the end of each training the descriptor is measured on the validation + test tiles at the operating thresholds and stored with the model. Training a model whose name already exists raises an error unless `overwrite=True`.

In [ ]:
TRAIN_DIR = "Data/images/ARCI URAS26F2"      # annotated tiles; masks in TRAIN_DIR/training_masks
TRAIN_SPEC = "ARCI URAS26F2"                 # name of the annotated set (start of the model name)
TRAINING_PARAMETERS = {}                     # e.g. {'seg_epochs': 200, 'class_epochs': 80}
SEG_MODEL_CONFIG = {}                        # e.g. {'threshold': 0.1}   operating point, stored with the model
CLASS_MODEL_CONFIG = {}                      # e.g. {'threshold': 0.55}

TRAINING_RUN = False

if TRAINING_RUN:
    seg_model, cls_model = OptimusPrimus.train_all(TRAIN_DIR, TRAIN_SPEC, training_parameters=TRAINING_PARAMETERS,
                                                   seg_model_config=SEG_MODEL_CONFIG, class_model_config=CLASS_MODEL_CONFIG)

### 1b. Manual route: classifier trained on hand-sorted patches

The same chain with a human in the middle. The two patch sources are never mixed: a classifier is trained either on the automatic patches or on a folder of sorted patches.

1. **Segmentation**: `train_segmentation` (cell below), same output as the first half of `train_all`.
2. **Export**: `export_candidate_patches(seg_model, images, max_patches=...)` runs the segmentation on any tiles (one plane per z-stack) and writes into `<model folder>/patches_manual_<images>/`:
   - `unsorted/<tile>__c<k>.png`: the 64×64 patch exactly as the classifier sees it (rotated, cropped on black);
   - `context/<tile>__c<k>.png`: the same candidate in its surroundings, outlined in green, to judge it by eye;
   - empty `track/` and `bkg/`, plus `candidates.csv` (source tile, position, length) and `export_info.json` (model, threshold).
3. **Sort by hand**: move each file of `unsorted/` into `track/` or `bkg/`, looking at the file with the same name in `context/`. Leave doubtful ones in `unsorted/`: they are not used. Do not edit the patches. To fix the validation set yourself, use `train/track`, `train/bkg`, `val/track`, `val/bkg` instead; otherwise 15 % is held out at random.
4. **Classifier**: `train_classifier(seg_model, patches=<folder>)` stores `<SEG_NAME>__cls-manual-<folder>-n<N>-<hash>_...pth` in the same model folder, with its descriptor.

Safeguards: the validation/test tiles of the segmentation model are never exported, and patches traced to them are refused at training (the descriptor is measured on them). Exporting into a folder that already holds patches, sorted ones included, is refused unless `overwrite=True`. A folder of patches made elsewhere (just `track/` and `bkg/`) is accepted as well.

`train_classifier(seg_model, 'auto')` trains the automatic classifier on an existing segmentation model.

In [ ]:
MANUAL_SEG = False

if MANUAL_SEG:
    seg_model = OptimusPrimus.train_segmentation(TRAIN_DIR, TRAIN_SPEC, training_parameters=TRAINING_PARAMETERS,
                                                 seg_model_config=SEG_MODEL_CONFIG)

In [ ]:
EXPORT_PATCHES = False
EXPORT_IMAGES = "Data/images/ARCI URAS26F2/tiles"   # any tiles: training tiles or unannotated acquisitions

if EXPORT_PATCHES:
    patch_dir = OptimusPrimus.export_candidate_patches(seg_model, EXPORT_IMAGES, max_patches=1000)

*Sort `unsorted/` into `track/` and `bkg/` now, then train:*

In [ ]:
MANUAL_CLS = False
PATCH_DIR = globals().get('patch_dir', "Data/models/<SEG_NAME>/patches_manual_<images>")

if MANUAL_CLS:
    cls_model = OptimusPrimus.train_classifier(seg_model, patches=PATCH_DIR, training_parameters=TRAINING_PARAMETERS,
                                               class_model_config=CLASS_MODEL_CONFIG)

### Available models

In [ ]:
OptimusPrimus.list_models()

## 2. Inference

`run_inference(images, seg_model, cls_model=None)`. With `cls_model=None` the run is **segmentation only**. Architecture, image size, pixel size and thresholds come from the records stored with the models (override with `seg_th=`, `cls_th=`). The tracks are saved in `Data/inference_results/<images>__<model>_st<t>[_ct<c>].csv`, and the descriptor of the same models and thresholds is attached to the result.

In [ ]:
IMAGES = "Data/images/ARCI URAS26F2/tiles"                                            # tiles to analyse
SEG_MODEL = globals().get('seg_model', "Data/models/<SEG_NAME>")                      # folder or .pth (mandatory)
CLS_MODEL = globals().get('cls_model', "Data/models/<SEG_NAME>/<SEG_NAME>__<CLS_TAG>.pth")   # None: segmentation only

result = OptimusPrimus.run_inference(IMAGES, SEG_MODEL, CLS_MODEL)
pd.Series(result.summary())

In [ ]:
# A saved run can be reloaded without running the networks:
# result = OptimusPrimus.load_inference("Data/inference_results/<run>.csv")
result.tracks.head()

## 3. Descriptor and detection model

The descriptor of the models used by `result` (measured on the annotated validation + test tiles), re-binned on the grid shared by theory, calibration and measurement. Per bin of **annotated** length: `G` annotated tracks, `TP`, `FN`, `recall`, `fn_rate`, and the length response of the matched pairs (`bias_nm` = mean reported − annotated, `sigma_nm`). Per bin of **reported** length: `Q`, `FP`, `precision`, `fp_per_cm2`.

The detection model is built from the table:
$$\mu_j=\sum_i M_{ji}\,R_i\,N_i+\phi_j\,A,$$
with $R_i$ = `recall_fold`, $M_{ji}$ a Gaussian migration of mean $L+$`bias_fold_nm` and width `sigma_fold_nm` from true bin $i$ to reported bin $j$, $\phi_j$ = `fp_per_cm2` and $A$ the analysed area. Bins with fewer than 5 annotated tracks (or matched pairs) take recall (or length response) from the nearest populated bin, flagged in `borrowed_recall` / `borrowed_response`. No calibration uncertainty is propagated: `G` shows where it would stop being negligible. The measurement carries its Poisson error.

In [ ]:
BIN_EDGES_NM = np.linspace(0., 16000., 17)
BIN_MIDS_UM = 0.5 * (BIN_EDGES_NM[1:] + BIN_EDGES_NM[:-1]) / 1e3

logs = OptimusPrimus.load_detection_logs(result.detection_logs)
descriptor = OptimusPrimus.detection_descriptor(logs, BIN_EDGES_NM)
print(f"{result.mode}: validation area {descriptor.attrs['area_cm2']:.4f} cm^2, {descriptor.attrs['n_pairs']} matched pairs "
      f"({descriptor.attrs['n_pairs_rejected']} gross length outliers excluded)")
descriptor.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(BIN_MIDS_UM, descriptor['recall'], 'o', label='recall (annotated length)')
ax[0].plot(BIN_MIDS_UM, descriptor['precision'], 's', label='precision (reported length)')
ax[0].set_ylim(0, 1.05); ax[0].legend()
ax[1].stairs(descriptor['fp_per_cm2'], BIN_EDGES_NM / 1e3)
ax[1].set_ylabel('false positives per cm$^2$')
pairs_um = np.asarray(logs['pair_log'])
ax[2].scatter(pairs_um[:, 0], pairs_um[:, 1] - pairs_um[:, 0], s=5, alpha=0.3, color='grey', label='matched pairs')
ax[2].errorbar(BIN_MIDS_UM, descriptor['bias_nm'] / 1e3, descriptor['sigma_nm'] / 1e3, fmt='ro', label='bias ± sigma')
ax[2].set_ylabel('reported − annotated [um]'); ax[2].legend()
for a in ax:
    a.set_xlabel('length [um]')
plt.tight_layout(); plt.show()

### Sliced theoretical spectrum

**Demo only**: `counts_mg` is a toy shape and the spectrum is rescaled to the number of tracks found. For the real analysis replace `counts_mg` with the PrimusCode spectrum (tracks per mg in each bin of `BIN_EDGES_NM`) and delete the rescaling line; `N_i` is then sliced on the analysed area.

In [ ]:
SAMPLE_DENSITY = 2.4                             # target density [g/cm^3]: set it for your sample (halite ~2.16)

x_mids = 0.5 * (BIN_EDGES_NM[1:] + BIN_EDGES_NM[:-1])
counts_mg = np.exp(-0.5 * ((x_mids - 6000.) / 1500.) ** 2)        # DEMO shape

# slice_spectrum is a Monte Carlo: average several realisations to suppress its own noise
N_i = np.mean([slice_spectrum(counts_mg, sample_area_cm2=result.area_cm2, sample_density_g_cm3=SAMPLE_DENSITY,
                              x_bins=BIN_EDGES_NM, angular_pdf=None, phi_cut_deg=0.,
                              pit_width=500., bulk_etching_depth=100., correction=True)
               for _ in range(50)], axis=0)
N_i = N_i * (len(result.tracks) / N_i.sum())     # DEMO normalisation: delete with the real spectrum
print(f"Sliced spectrum: {N_i.sum():.0f} tracks in {len(N_i)} bins")

### Fold and compare

Poisson likelihood-ratio $\chi^2$ (valid at low counts, no bin merging). With a normalisation fitted to the data, as in the demo, subtract one degree of freedom.

In [ ]:
model = OptimusPrimus.detection_model(descriptor)
mu, mu_tracks, mu_fp = OptimusPrimus.fold_detection(N_i, model, result.area_cm2, return_parts=True)
n_obs = result.histogram(BIN_EDGES_NM)
chi2, ndof, pval = OptimusPrimus.poisson_chi2(n_obs, mu)
print(f"chi2 / ndof = {chi2:.1f} / {ndof}   p = {pval:.3f}   (expected {mu.sum():.0f}, observed {n_obs.sum()})")

edges_um = BIN_EDGES_NM / 1e3
fig, ax = plt.subplots(figsize=(7, 4))
for i in np.where(descriptor['borrowed_recall'])[0]:
    ax.axvspan(edges_um[i], edges_um[i + 1], color='0.92', lw=0)
ax.stairs(mu, edges_um, color='C0', label='expected')
ax.stairs(mu_fp, edges_um, color='C1', ls='--', label='of which false positives')
ax.errorbar(BIN_MIDS_UM, n_obs, np.sqrt(np.maximum(n_obs, 1)), fmt='ko', ms=4, label='measured (Poisson)')
ax.set_xlabel('track length [um]'); ax.set_ylabel('tracks per bin'); ax.legend()
ax.set_title('grey: recall borrowed from a neighbouring bin', fontsize=9)
plt.tight_layout(); plt.show()

### Other operating points

Thresholds are part of the model description. To use different ones, first measure the descriptor at those thresholds, then run inference with the same values:
```python
OptimusPrimus.describe_model(SEG_MODEL, CLS_MODEL, cls_th=0.7)
result = OptimusPrimus.run_inference(IMAGES, SEG_MODEL, CLS_MODEL, cls_th=0.7)
```